In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Charger le dataset augmenté
df = pd.read_csv('../data/processed/reclamations_clean_augmente.csv')
print(f"Dataset : {len(df)} réclamations")
print(df['categorie'].value_counts())

# Préparer X et y
X = df['texte_clean']
y = df['categorie']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline améliorée
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=8000,        # Plus de features
        ngram_range=(1, 3),       # Mots seuls, paires, triplets
        min_df=2,
        max_df=0.95,
        sublinear_tf=True         # Meilleur poids des fréquences
    )),
    ('classifier', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42,
        C=1.0
    ))
])

# Entraîner
print("\n🎓 Entraînement en cours...")
pipeline.fit(X_train, y_train)

# Évaluer
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n🎯 Nouvelle accuracy : {accuracy*100:.2f}%")

# Rapport détaillé
print("\n📊 Rapport :")
print(classification_report(y_test, y_pred))

# Sauvegarder
joblib.dump(pipeline, '../models/classifier_pipeline.pkl')
joblib.dump(list(pipeline.classes_), '../models/categories.pkl')
print("\n✅ Nouveau modèle sauvegardé !")

C:\Users\suki\AppData\Local\Temp\ipykernel_27696\1880923704.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Dataset : 1068 réclamations
categorie
retard_livraison         226
mauvaise_qualite         195
produit_casse            191
probleme_transport       185
erreur_picking            98
erreur_administrative     95
article_manquant          78
Name: count, dtype: int64

🎓 Entraînement en cours...

🎯 Nouvelle accuracy : 77.10%

📊 Rapport :
                       precision    recall  f1-score   support

     article_manquant       0.68      0.94      0.79        16
erreur_administrative       0.70      0.84      0.76        19
       erreur_picking       1.00      0.65      0.79        20
     mauvaise_qualite       0.69      0.87      0.77        39
   probleme_transport       0.84      0.86      0.85        37
        produit_casse       0.78      0.55      0.65        38
     retard_livraison       0.81      0.76      0.78        45

             accuracy                           0.77       214
            macro avg       0.79      0.78      0.77       214
         weighted avg       0.

In [2]:
import joblib

modele = joblib.load('../models/classifier_pipeline.pkl')

tests = [
    "pas encore arrivé",
    "j'ai lancé cette commande ça fait longtemps mais pas encore reçu",
    "Mes médicaments sont arrivés cassés et en retard, c'est urgent",
    "J'ai reçu un mauvais ordinateur, ce n'est pas le modèle que j'avais commandé",
    "Il manque la moitié de ma commande",
    "La qualité du produit est très décevante",
    "Le livreur n'a pas trouvé mon adresse",
]

print(f"{'Texte':<70} {'Prédit':<25} {'Conf'}")
print("-" * 110)

for texte in tests:
    # Appliquer le preprocessing
    texte_clean = preprocesser(texte)
    pred = modele.predict([texte_clean])[0]
    probas = modele.predict_proba([texte_clean])[0]
    conf = max(probas) * 100
    
    print(f"{texte[:68]:<70} {pred:<25} {conf:5.1f}%")

Texte                                                                  Prédit                    Conf
--------------------------------------------------------------------------------------------------------------


NameError: name 'preprocesser' is not defined